In [17]:
import numpy as np
import pandas as pd
from nltk.tokenize import sent_tokenize, word_tokenize
import string
import ast
import re
import os
from pathlib import Path

## Data Cleaning

In [11]:
# Using os.path - go up one directory and access the file
data_path = os.path.join(os.path.dirname(os.path.abspath('')), 'job_descriptions.csv')
df = pd.read_csv(data_path)

In [30]:
# drop the url since it is not a relevant predictor
clean_df = df.drop('url', axis=1)

# split data into jobs and skills
jobs = clean_df['job']                  
skills = clean_df['skills']

### Job Description

We tokenize by word and add a `job_id` column.

In [31]:
# Standardize punctuation to prevent random biases.

punkt = string.punctuation
punkt = punkt[0:2] + punkt[3:13] + punkt[14:-1]

for i in range(len(jobs)):
    jobs.loc[i] = jobs.loc[i].lower()

    punkt_str = jobs.loc[i]
    test_str = punkt_str.translate(str.maketrans('', '', punkt))
    jobs.loc[i] = test_str

'summarythe database developer is part of the cemco development team whose responsibilities include managing and maintaining the enterprise data warehouse. strong development skills in c# and .net including excel vsto. the ideal candidate should have a deep understanding of database management systems as well as the ability to write complex code in c# and .net to build maintain and optimize databases. effective at communicating with users to define requirements and design solutions based on those requirements.this position is also responsible for creating tools that provide insight to various departments including finance accounting and customer service to build data visualizations and dashboards.essential duties and responsibilitiesdesign install configure and maintain database management systems.monitor database performance and provide optimization.develop implement and maintain backup and recovery procedures.perform database security administration.ensure data integrity reliability 

In [22]:
for i in range(len(jobs)):
    jobs.loc[i] = word_tokenize(jobs[i])

jobs = jobs.explode(ignore_index=False).rename_axis('job_id').reset_index()

In [ ]:
jopp

### Skills

The skills data is in a `string` format, thus we convert from a string to a list using this claude generated function.

In [23]:
# Standardize punctuation to prevent random biases.
punkt_skills = punkt[:10] + punkt[11:]

for i in range(len(skills)):
    skills.loc[i] = skills.loc[i].lower()

    punkt_str = skills.loc[i]
    test_str = punkt_str.translate(str.maketrans('', '', punkt_skills))
    skills.loc[i] = test_str

In [24]:
# Assuming df is your dataframe and 'skills' is your column name
def convert_string_to_list(skills_str):
    try:
        # For properly formatted strings like "[item1, item2, item3]"
        return ast.literal_eval(skills_str)
    except (ValueError, SyntaxError):
        # Handle potential errors or edge cases
        if isinstance(skills_str, list):
            # If it's already a list, return it as is
            return skills_str
        elif isinstance(skills_str, str):
            # For simple comma-separated strings without brackets
            if skills_str.strip() == '':
                return []
            if '[' not in skills_str and ']' not in skills_str:
                return [item.strip() for item in skills_str.split(',')]
        # Default fallback
        return []

# Apply the conversion to the skills column
skills = skills.apply(convert_string_to_list)

In [25]:
def split_and_tag(skills_list):
    """
    Split a skills list into single words, then give them the IOB tags.
    
    """

    split_words = []
    tags = []
    
    for skill in skills_list:
        words = skill.split()
        split_words.extend(words)
        
        if len(words) == 1:
            tags.append('B')
        else:
            # First word gets 'B', subsequent words get 'I'
            tags.append('B')
            tags.extend(['I'] * (len(words) - 1))
    
    skill_tag_dict = {
        'skills' : split_words,
        'tags' : tags
    }

    return skill_tag_dict

### Adding features

We conduct IOB tagging to add further features into our data increasing the performance of our model.

In [26]:
def get_tag(curr_dict: dict, search_str: str):
    if search_str in curr_dict['skills']:
        idx = curr_dict['skills'].index(search_str)
        tag = curr_dict['tags'][idx]
        return tag
    else:
        return 'O'

In [27]:
jd = jobs[jobs['job_id'] == 0]['job']

for word in jd:
    print(word)

summarythe
database
developer
is
part
of
the
cemco
development
team
whose
responsibilities
include
managing
and
maintaining
the
enterprise
data
warehouse
.
strong
development
skills
in
c
#
and
.net
including
excel
vsto
.
the
ideal
candidate
should
have
a
deep
understanding
of
database
management
systems
as
well
as
the
ability
to
write
complex
code
in
c
#
and
.net
to
build
maintain
and
optimize
databases
.
effective
at
communicating
with
users
to
define
requirements
and
design
solutions
based
on
those
requirements.this
position
is
also
responsible
for
creating
tools
that
provide
insight
to
various
departments
including
finance
accounting
and
customer
service
to
build
data
visualizations
and
dashboards.essential
duties
and
responsibilitiesdesign
install
configure
and
maintain
database
management
systems.monitor
database
performance
and
provide
optimization.develop
implement
and
maintain
backup
and
recovery
procedures.perform
database
security
administration.ensure
data
integrity
reliabil

In [29]:
skills[0]

['c#',
 '.net',
 'database performance',
 'c#',
 '.net',
 'tsql',
 'sql',
 'project management',
 'collaboration',
 'teamwork',
 'close vision',
 'distance vision',
 'reasonable accommodations',
 'office',
 'cemco',
 'llc',
 'cemco',
 'steel fram']